# 09 Paper Analysis Source

Converted from `code/paper_analysis.py` for Colab/code review. Edit/comment cells here, then save the reviewed notebook.


Run this notebook from the repository root after cloning the project in Colab. The production script remains in `code/`; this notebook is a review/editing copy.

In [ ]:
# Optional Colab setup
from pathlib import Path
import os

if Path('requirements.txt').exists():
    print('Repository root detected:', Path.cwd())
else:
    print('Clone/cd into the repository root before running script cells.')


In [ ]:
#!/usr/bin/env python3
"""Paper-ready analysis outputs: tuned models, regressions, SHAP figures, and captions."""

from __future__ import annotations

import json
import os
import re
import argparse
from pathlib import Path
from typing import Any

# Set environment variables for cache directories to manage storage # enext
os.environ.setdefault("MPLCONFIGDIR", str(Path(".cache/matplotlib").resolve()))
os.environ.setdefault("PYBASEBALL_CACHE", str(Path(".cache/pybaseball").resolve()))
os.environ.setdefault("XDG_CACHE_HOME", str(Path(".cache").resolve()))

import matplotlib

# Use 'Agg' backend for matplotlib to avoid GUI issues in non-interactive environments # enext
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import statsmodels.formula.api as smf
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Import custom modules for paths and pitch labeling # enext
from pitch_sequence_pipeline import PROCESSED_DIR, ROOT, log, pitch_label, sequence_features_path, temporal_sequence_features_path


# Define output directories for organized storage of results # enext
OUTPUT_DIR = ROOT / "output"
PAPER_DIR = OUTPUT_DIR / "paper_ready"
TABLE_DIR = PAPER_DIR / "tables"
FIGURE_DIR = PAPER_DIR / "figures"

# List of categorical features used in models # enext
MODEL_CAT = [
    "pitcher_name",
    "batter_name",
    "stand",
    "p_throws",
    "count",
    "base_state",
    "prev_pitch_type",
    "prev2_pitch_type",
    "prev_description",
    "inning_topbot",
]

# List of numerical features used in models # enext
MODEL_NUM = [
    "balls",
    "strikes",
    "outs_when_up",
    "inning",
    "pitch_number",
    "prev_zone",
    "prev_release_speed",
    "prev_plate_x",
    "prev_plate_z",
    "score_diff_batting_team",
    "lineup_left_share",
    "lineup_right_share",
    "lineup_switch_share",
    "lineup_batter_count",
]

# Specifications for state-machine models, defining different key combinations # enext
STATE_SPECS = {
    "pitcher_batter_count_prev_stand": ["pitcher_name", "batter_name", "count", "prev_pitch_type", "stand"],
    "pitcher_count_prev_stand": ["pitcher_name", "count", "prev_pitch_type", "stand"],
}

# Fallback keys for state-machine models if primary keys are not available or sufficient # enext
STATE_FALLBACKS = [
    ["pitcher_name", "count", "prev_pitch_type"],
    ["pitcher_name", "batter_name", "count"],
    ["pitcher_name", "count"],
    ["pitcher_name", "prev_pitch_type"],
    ["pitcher_name"],
]

In [ ]:
def ensure_dirs() -> None:
    # Create necessary output directories if they don't exist # enext
    for path in [PAPER_DIR, TABLE_DIR, FIGURE_DIR]:
        path.mkdir(parents=True, exist_ok=True)

In [ ]:
def set_style() -> None:
    # Set aesthetic style for seaborn plots # enext
    sns.set_theme(style="whitegrid", context="paper")
    # Update matplotlib parameters for consistent figure appearance # enext
    plt.rcParams.update(
        {
            "figure.dpi": 140,
            "savefig.dpi": 300,
            "font.family": "DejaVu Serif",
            "axes.titleweight": "bold",
            "axes.labelsize": 10,
            "axes.titlesize": 12,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 9,
            "legend.title_fontsize": 9,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )

In [ ]:
def save_fig(name: str) -> Path:
    path = FIGURE_DIR / name
    # Adjust plot layout to prevent labels from overlapping # enext
    plt.tight_layout()
    # Save the figure to the specified path with tight bounding box # enext
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    # Log the path of the saved figure # enext
    log(f"wrote {path.relative_to(ROOT)}")
    return path

In [ ]:
def load_features(top_n: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Load 2025 feature data from a parquet file # enext
    df_2025 = pd.read_parquet(sequence_features_path(2025, top_n))
    # Construct path for 2026 temporal feature data # enext
    path_2026 = temporal_sequence_features_path(2025, 2026, top_n)
    # Load 2026 data if the file exists, otherwise create an empty DataFrame # enext
    df_2026 = pd.read_parquet(path_2026) if path_2026.exists() else pd.DataFrame()
    return df_2025, df_2026

In [ ]:
def model_frame(
    df: pd.DataFrame,
    keep_pitch_types: pd.Index | None = None,
    min_pitch_type_count: int = 1,
) -> pd.DataFrame:
    # Create a copy of the DataFrame, filtering out rows with missing 'pitch_type' # enext
    out = df.loc[df["pitch_type"].notna()].copy()
    if out.empty:
        return out
    # If no specific pitch types to keep are provided, determine them based on counts # enext
    if keep_pitch_types is None:
        counts = out["pitch_type"].value_counts()
        # Keep pitch types that meet the minimum count threshold # enext
        keep_pitch_types = counts.loc[counts >= min_pitch_type_count].index
    # Filter the DataFrame to include only the specified or determined pitch types # enext
    return out.loc[out["pitch_type"].isin(keep_pitch_types)].copy()

In [ ]:
def available_columns(df: pd.DataFrame, columns: list[str]) -> list[str]:
    # Return a list of columns that exist in the DataFrame from a given list # enext
    return [col for col in columns if col in df.columns]

In [ ]:
def top_k_accuracy(classes: np.ndarray, probs: np.ndarray, y: pd.Series, k: int) -> float:
    # Get the indices that would sort the probabilities in ascending order # enext
    order = np.argsort(probs, axis=1)[:, -min(k, len(classes)) :]
    # Select the top k classes based on sorted probabilities # enext
    top = classes[order]
    # Calculate the mean accuracy where the actual class is within the top k predicted classes # enext
    return float(np.mean([actual in row for actual, row in zip(y.to_numpy(), top)]))

In [ ]:
def evaluate_probs(model: str, dataset: str, classes: np.ndarray, probs: np.ndarray, y: pd.Series) -> dict[str, Any]:
    # Predict the class with the highest probability # enext
    pred = classes[np.argmax(probs, axis=1)]
    # Create a mapping from pitch type to its index in the classes array # enext
    class_to_idx = {pitch: idx for idx, pitch in enumerate(classes)}
    # Get the probability of the actual pitch type # enext
    actual_prob = np.array([probs[i, class_to_idx.get(pitch, -1)] if pitch in class_to_idx else 0.0 for i, pitch in enumerate(y)])
    # Calculate log loss, handling cases where only one unique class is present # enext
    loss = log_loss(y, probs, labels=list(classes)) if y.nunique() > 1 else np.nan
    return {
        "model": model,
        "dataset": dataset,
        "rows": int(len(y)),
        "exact_accuracy": float(accuracy_score(y, pred)), # enext
        "top2_accuracy": top_k_accuracy(classes, probs, y, 2), # enext
        "top3_accuracy": top_k_accuracy(classes, probs, y, 3), # enext
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)), # enext
        "avg_probability_actual_pitch": float(actual_prob.mean()), # enext
        "avg_model_confidence": float(probs.max(axis=1).mean()), # enext
        "log_loss": float(loss), # enext
    }

In [ ]:
def build_arsenal(train_rows: pd.DataFrame, min_usage: float = 0.03, min_pitches: int = 30) -> pd.DataFrame:
    # Define aggregation specifications for pitch metrics # enext
    agg_spec: dict[str, tuple[str, str]] = {
        "pitches": ("pitch_type", "size"),
        "avg_velocity": ("release_speed", "mean"),
        "whiff_rate": ("whiff", "mean"),
        "weak_contact_rate": ("weak_contact", "mean"),
        "mean_pitcher_run_value": ("pitcher_run_value", "mean"),
    }
    # Group by pitcher and pitch type to calculate arsenal statistics # enext
    arsenal = (
        train_rows.groupby(["pitcher_name", "pitch_type"], as_index=False)
        .agg(**agg_spec)
        .sort_values(["pitcher_name", "pitches"], ascending=[True, False])
    )
    # Map pitch types to their labels # enext
    arsenal["pitch_label"] = arsenal["pitch_type"].map(pitch_label)
    # Calculate total pitches for each pitcher # enext
    arsenal["pitcher_total_pitches"] = arsenal.groupby("pitcher_name")["pitches"].transform("sum")
    # Calculate usage rate for each pitch type # enext
    arsenal["usage_rate"] = arsenal["pitches"] / arsenal["pitcher_total_pitches"]
    # Determine if a pitch is 'in arsenal' based on usage rate or minimum pitches # enext
    arsenal["in_arsenal"] = (arsenal["usage_rate"] >= min_usage) | (arsenal["pitches"] >= min_pitches)
    # Ensure the most frequently thrown pitch is always considered 'in arsenal' # enext
    top_idx = arsenal.groupby("pitcher_name")["pitches"].idxmax()
    arsenal.loc[top_idx, "in_arsenal"] = True
    # Calculate the size of each pitcher's arsenal # enext
    arsenal["arsenal_size"] = arsenal.groupby("pitcher_name")["in_arsenal"].transform("sum").astype(int)
    # Save the full and filtered arsenal data to CSV files # enext
    arsenal.to_csv(TABLE_DIR / "pitcher_arsenals.csv", index=False)
    arsenal.loc[arsenal["in_arsenal"]].to_csv(TABLE_DIR / "pitcher_arsenals_allowed_only.csv", index=False)
    return arsenal

In [ ]:
def arsenal_map(arsenal: pd.DataFrame) -> dict[str, set[str]]:
    # Create a dictionary mapping each pitcher to a set of their 'in-arsenal' pitch types # enext
    return arsenal.loc[arsenal["in_arsenal"]].groupby("pitcher_name")["pitch_type"].apply(lambda s: set(s.astype(str))).to_dict()

In [ ]:
def apply_arsenal_mask(
    probs: np.ndarray,
    pitcher_names: pd.Series,
    classes: np.ndarray,
    allowed: dict[str, set[str]],
) -> np.ndarray:
    # Create a mapping from pitch type to its index in the classes array # enext
    idx = {pitch: i for i, pitch in enumerate(classes)}
    # Initialize an array for masked probabilities with zeros # enext
    masked = np.zeros_like(probs)
    # Iterate through each prediction to apply arsenal masking # enext
    for i, pitcher_name in enumerate(pitcher_names.astype(str)):
        # Get allowed pitches for the current pitcher # enext
        allowed_pitches = allowed.get(pitcher_name)
        # If no allowed pitches, use original probabilities # enext
        if not allowed_pitches:
            masked[i] = probs[i]
            continue
        # Get indices of allowed pitches # enext
        allowed_idx = [idx[pitch] for pitch in allowed_pitches if pitch in idx]
        # If no allowed indices, use original probabilities # enext
        if not allowed_idx:
            masked[i] = probs[i]
            continue
        # Create a row of probabilities, setting non-allowed pitches to zero # enext
        row = np.zeros(probs.shape[1])
        row[allowed_idx] = probs[i, allowed_idx]
        total = row.sum()
        # Normalize probabilities for allowed pitches # enext
        if total > 0:
            row /= total
        else:
            # If total is zero, distribute probability equally among allowed pitches # enext
            row[allowed_idx] = 1 / len(allowed_idx)
        masked[i] = row
    return masked

In [ ]:
def state_machine_probs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    classes: np.ndarray,
    keys: list[str],
    alpha: float,
    min_count: int,
) -> np.ndarray:
    # Calculate global pitch type probabilities from training data # enext
    global_counts = train["pitch_type"].value_counts().reindex(classes, fill_value=0).astype(float)
    global_probs = ((global_counts + alpha) / (global_counts.sum() + alpha * len(classes))).to_numpy()
    tables = []
    # Iterate through key sets (including fallbacks) to build state tables # enext
    for key_set in [keys] + STATE_FALLBACKS:
        # Filter key set to include only columns present in both train and test # enext
        key_set = [col for col in key_set if col in train.columns and col in test.columns]
        if not key_set:
            continue
        # Calculate pitch type counts for each state # enext
        counts = train.groupby(key_set + ["pitch_type"]).size().unstack(fill_value=0)
        counts = counts.reindex(columns=classes, fill_value=0).astype(float)
        totals = counts.sum(axis=1)
        # Calculate conditional probabilities with Laplace smoothing # enext
        probs = (counts + alpha).div(totals + alpha * len(classes), axis=0)
        lookup = {}
        for key, row in probs.iterrows():
            # Only use states with a minimum count of observations # enext
            if totals.loc[key] >= min_count:
                lookup[key] = row.to_numpy(dtype=float)
        tables.append((key_set, lookup))

    # Initialize output array for probabilities # enext
    out = np.zeros((len(test), len(classes)))
    # Iterate through test rows to assign probabilities based on state # enext
    for i, row in enumerate(test.itertuples(index=False)):
        row_map = row._asdict()
        chosen = None
        # Find the most specific state for which probabilities are available # enext
        for key_set, probs_table in tables:
            key = tuple(row_map[col] for col in key_set)
            if len(key) == 1:
                key = key[0]
            chosen = probs_table.get(key)
            if chosen is not None:
                break
        # Use chosen state probabilities or global probabilities if no state is found # enext
        out[i] = chosen if chosen is not None else global_probs
    return out

In [ ]:
def tune_state_machine(
    train: pd.DataFrame,
    test: pd.DataFrame,
    classes: np.ndarray,
    allowed: dict[str, set[str]],
    eval_rows: pd.DataFrame | None = None,
) -> pd.DataFrame:
    rows = []
    # Define evaluation sets, including 2025 holdout and optionally 2026 data # enext
    eval_sets = [("2025_holdout", test)]
    if eval_rows is not None and not eval_rows.empty:
        eval_sets.append(("2026_to_date", eval_rows))

    # Iterate through state specifications and tuning parameters # enext
    for spec_name, keys in STATE_SPECS.items():
        for alpha in [1.0]: # Laplace smoothing parameter # enext
            for min_count in [5, 15]: # Minimum observation count for a state # enext
                log(f"  state grid {spec_name}, alpha={alpha}, min_count={min_count}")
                for dataset_name, rows_eval in eval_sets:
                    # Calculate probabilities using the state-machine model # enext
                    probs = state_machine_probs(train, rows_eval, classes, keys, alpha, min_count)
                    # Evaluate the model performance without arsenal masking # enext
                    rows.append(
                        {
                            **evaluate_probs(f"state:{spec_name}", dataset_name, classes, probs, rows_eval["pitch_type"]),
                            "alpha": alpha,
                            "min_count": min_count,
                            "state_spec": spec_name,
                            "arsenal_masked": False,
                        }
                    )
                    # Apply arsenal masking and re-evaluate # enext
                    masked = apply_arsenal_mask(probs, rows_eval["pitcher_name"], classes, allowed)
                    rows.append(
                        {
                            **evaluate_probs(
                                f"state:{spec_name}:arsenal_masked",
                                dataset_name,
                                classes,
                                masked,
                                rows_eval["pitcher_name"],
                            ),
                            "alpha": alpha,
                            "min_count": min_count,
                            "state_spec": spec_name,
                            "arsenal_masked": True,
                        }
                    )
    # Create a DataFrame from the results and sort them # enext
    out = pd.DataFrame(rows).sort_values(["dataset", "exact_accuracy", "top3_accuracy"], ascending=[True, False, False])
    # Save the tuning grid results to a CSV file # enext
    out.to_csv(TABLE_DIR / "state_machine_tuning_grid.csv", index=False)
    return out

In [ ]:
def make_hgb_model(cat_cols: list[str], num_cols: list[str], params: dict[str, Any]) -> Pipeline:
    # Define preprocessing steps for categorical features # enext
    cat_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]
    )
    # Define preprocessing steps for numerical features # enext
    num_pipe = Pipeline([("impute", SimpleImputer(strategy="median"))])
    # Combine preprocessors using ColumnTransformer # enext
    pre = ColumnTransformer([("cat", cat_pipe, cat_cols), ("num", num_pipe, num_cols)])
    # Create and return a full pipeline including preprocessing and HistGradientBoostingClassifier # enext
    return Pipeline(
        [
            ("preprocess", pre),
            (
                "model",
                HistGradientBoostingClassifier(
                    random_state=42,
                    **params,
                ),
            ),
        ]
    )

In [ ]:
def make_logistic_model(cat_cols: list[str], num_cols: list[str]) -> Pipeline:
    # Define preprocessing steps for categorical features # enext
    cat_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
        ]
    )
    # Define preprocessing steps for numerical features # enext
    num_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler(with_mean=False)),
        ]
    )
    # Combine preprocessors using ColumnTransformer # enext
    pre = ColumnTransformer([("cat", cat_pipe, cat_cols), ("num", num_pipe, num_cols)])
    # Create and return a full pipeline including preprocessing and LogisticRegression # enext
    return Pipeline(
        [
            ("preprocess", pre),
            ("model", LogisticRegression(C=0.35, max_iter=1000, solver="lbfgs")),
        ]
    )

In [ ]:
def model_probs(model: Pipeline, X: pd.DataFrame, classes: np.ndarray) -> np.ndarray:
    # Get raw probability predictions from the model # enext
    raw = model.predict_proba(X)
    # Initialize output array with zeros # enext
    out = np.zeros((len(X), len(classes)))
    # Create a mapping from model's classes to the universal classes array # enext
    local_idx = {pitch: i for i, pitch in enumerate(model.classes_)}
    # Populate the output array, aligning model predictions with universal classes # enext
    for j, pitch in enumerate(classes):
        if pitch in local_idx:
            out[:, j] = raw[:, local_idx[pitch]]
    # Calculate row sums to identify missing predictions # enext
    row_sum = out.sum(axis=1)
    missing = row_sum == 0
    # For rows with missing predictions, assign equal probability to all classes # enext
    out[missing] = 1 / len(classes)
    # Normalize probabilities for rows that had predictions # enext
    out[~missing] = out[~missing] / row_sum[~missing, None]
    return out

In [ ]:
def tune_models(
    train: pd.DataFrame,
    test: pd.DataFrame,
    classes: np.ndarray,
    allowed: dict[str, set[str]],
    eval_rows: pd.DataFrame | None = None,
) -> pd.DataFrame:
    # Get available categorical and numerical columns from the training data # enext
    cat_cols = available_columns(train, MODEL_CAT)
    num_cols = available_columns(train, MODEL_NUM)
    features = cat_cols + num_cols
    rows = []
    # Define evaluation sets, including 2025 holdout and optionally 2026 data # enext
    eval_sets = [("2025_holdout", test)]
    if eval_rows is not None and not eval_rows.empty:
        eval_sets.append(("2026_to_date", eval_rows))

    # Define a grid of hyperparameters for HistGradientBoostingClassifier # enext
    grid = [
        {"learning_rate": 0.07, "max_iter": 100, "max_leaf_nodes": 31, "l2_regularization": 0.02},
        {"learning_rate": 0.07, "max_iter": 180, "max_leaf_nodes": 31, "l2_regularization": 0.02},
        {"learning_rate": 0.05, "max_iter": 180, "max_leaf_nodes": 31, "l2_regularization": 0.02},
        {"learning_rate": 0.08, "max_iter": 160, "max_leaf_nodes": 31, "l2_regularization": 0.05},
    ]
    # Iterate through the hyperparameter grid to tune models # enext
    for i, params in enumerate(grid, start=1):
        name = f"pooled_hgb_tuned_{i}"
        log(f"  training {name}: {params}")
        # Create and train the HGB model with current parameters # enext
        model = make_hgb_model(cat_cols, num_cols, params)
        model.fit(train[features], train["pitch_type"])
        for dataset_name, rows_eval in eval_sets:
            # Get model probabilities # enext
            probs = model_probs(model, rows_eval[features], classes)
            # Evaluate performance without arsenal masking # enext
            row = evaluate_probs(name, dataset_name, classes, probs, rows_eval["pitch_type"])
            row.update(params)
            rows.append(row)
            # Apply arsenal masking and re-evaluate # enext
            masked = apply_arsenal_mask(probs, rows_eval["pitcher_name"], classes, allowed)
            masked_row = evaluate_probs(f"{name}_arsenal_masked", dataset_name, classes, masked, rows_eval["pitcher_name"])
            masked_row.update(params)
            rows.append(masked_row)

    # Create a DataFrame from the results and sort them # enext
    out = pd.DataFrame(rows).sort_values(["dataset", "exact_accuracy", "top3_accuracy"], ascending=[True, False, False])
    # Save the tuning grid results to a CSV file # enext
    out.to_csv(TABLE_DIR / "pooled_model_tuning_grid.csv", index=False)
    return out

In [ ]:
def fit_regression_tables(rows: pd.DataFrame, target_pitches: list[str]) -> pd.DataFrame:
    model_rows = rows.copy()
    # Define the right-hand side of the regression formula, using categorical variables with reference levels # enext
    formula_rhs = 'C(count, Treatment(reference="0-0")) + C(prev_pitch_type, Treatment(reference="START")) + C(stand, Treatment(reference="L")) + C(pitcher_name)'
    coef_rows = []
    # Iterate through target pitches to fit separate OLS models # enext
    for pitch in target_pitches:
        target = f"is_{pitch}"
        # Create a binary target variable for the current pitch type # enext
        model_rows[target] = model_rows["pitch_type"].eq(pitch).astype(int)
        # Fit OLS model with robust standard errors (HC1) # enext
        model = smf.ols(f"{target} ~ {formula_rhs}", data=model_rows).fit(cov_type="HC1")
        # Extract coefficients and related statistics # enext
        for term, beta in model.params.items():
            coef_rows.append(
                {
                    "target_pitch_type": pitch,
                    "target_pitch_label": pitch_label(pitch),
                    "term": term,
                    "beta": float(beta),
                    "std_error": float(model.bse[term]),
                    "p_value": float(model.pvalues[term]),
                    "n_obs": int(model.nobs),
                    "r_squared": float(model.rsquared),
                }
            )
    # Create a DataFrame from the collected coefficients # enext
    out = pd.DataFrame(coef_rows)
    # Save the regression results to a CSV file # enext
    out.to_csv(TABLE_DIR / "ols_pooled_fixed_effects_coefficients.csv", index=False)
    return out

In [ ]:
def readable_term(term: str) -> str:
    if term == "Intercept":
        return "Intercept"
    # Use regular expressions to make regression terms more readable # enext
    term = re.sub(r'C\(count, Treatment\(reference="0-0"\)\)\[T\.(.+?)\]', r"Count: \1", term)
    term = re.sub(r'C\(prev_pitch_type, Treatment\(reference="START"\)\)\[T\.(.+?)\]', r"Previous pitch: \1", term)
    term = re.sub(r'C\(stand, Treatment\(reference="L"\)\)\[T\.(.+?)\]', r"Batter side: \1", term)
    term = re.sub(r"C\(pitcher_name\)\[T\.(.+?)\]", r"Pitcher: \1", term)
    return term

In [ ]:
def clean_feature_name(name: str) -> str:
    # Remove internal prefixes from feature names # enext
    name = name.replace("cat__", "").replace("num__", "")
    # Define mappings for common feature prefixes to more readable labels # enext
    mappings = {
        "pitcher_name_": "Pitcher: ",
        "count_": "Count: ",
        "prev_pitch_type_": "Previous pitch: ",
        "prev2_pitch_type_": "Two pitches ago: ",
        "stand_": "Batter side: ",
        "prev_description_": "Previous result: ",
        "inning_topbot_": "Half inning: ",
        "base_state_": "Base state: ",
    }
    # Apply mappings to clean feature names # enext
    for prefix, label in mappings.items():
        if name.startswith(prefix):
            return label + name[len(prefix) :]
    # Replace underscores with spaces for any remaining features # enext
    return name.replace("_", " ")

In [ ]:
def shap_feature_group(feature: str) -> str:
    # Group SHAP features into broader conceptual categories for easier interpretation # enext
    if feature.startswith("Pitcher: "):
        return "Pitcher identity / arsenal"
    if feature.startswith("Previous pitch: "):
        return "Previous pitch type"
    if feature.startswith("Two pitches ago: "):
        return "Two-pitches-ago pitch type"
    if feature.startswith("Batter side: "):
        return "Batter handedness"
    if feature.startswith("Count: ") or feature in {"balls", "strikes"}:
        return "Count / ball-strike leverage"
    if feature.startswith("Previous result: "):
        return "Previous pitch result"
    if feature.startswith("Base state: "):
        return "Base/out state"
    if feature in {"prev release speed", "pitch number", "score diff batting team"}:
        return feature
    return "Other context"

In [ ]:
def compact_model_label(name: str) -> str:
    # Provide more concise and readable labels for different models # enext
    if name == "logistic_lpm_style_baseline":
        return "Logistic baseline"
    if name == "logistic_lpm_style_baseline_arsenal_masked":
        return "Logistic + arsenal"
    if name.startswith("state:"):
        label = "State machine"
        if name.endswith(":arsenal_masked"):
            label += " + arsenal"
        return label
    if name.startswith("pooled_hgb"):
        label = "Pooled boosted trees"
        if name.endswith("_arsenal_masked"):
            label += " + arsenal"
        return label
    return name

In [ ]:
def make_paper_figures(
    arsenal: pd.DataFrame,
    state_grid: pd.DataFrame,
    model_grid: pd.DataFrame,
    regression: pd.DataFrame,
) -> None:
    # Figure 1: Pitcher-specific arsenals # enext
    allowed = arsenal.loc[arsenal["in_arsenal"]].copy()
    # Pivot table to show usage rate of each pitch type per pitcher # enext
    pivot = allowed.pivot_table(index="pitcher_name", columns="pitch_type", values="usage_rate", fill_value=0)
    # Order pitchers by arsenal size # enext
    order = allowed.groupby("pitcher_name")["arsenal_size"].max().sort_values(ascending=True).index
    pivot = pivot.reindex(order)
    # Create a stacked bar chart of pitch usage # enext
    ax = pivot.plot(kind="barh", stacked=True, figsize=(7.2, 6.2), width=0.82, colormap="tab20")
    ax.set_title("Pitcher-specific arsenals define the strategic menu")
    ax.set_xlabel("Pitch usage share")
    ax.set_ylabel("")
    ax.legend(title="Pitch", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    save_fig("fig1_pitcher_specific_arsenals.png")

    # Figure 2: Model comparison # enext
    state_2025 = state_grid.loc[state_grid["dataset"].eq("2025_holdout")].copy()
    model_2025 = model_grid.loc[model_grid["dataset"].eq("2025_holdout")].copy()
    # Get the names of the best performing state-machine and pooled models # enext
    best_state_name = state_2025.iloc[0]["model"]
    best_model_name = model_2025.iloc[0]["model"]
    # Define models to include in the comparison plot # enext
    comparison_names = [
        "logistic_lpm_style_baseline",
        "logistic_lpm_style_baseline_arsenal_masked",
        best_state_name,
        best_model_name,
    ]
    # Add the unmasked version of the best model if it's masked # enext
    if str(best_model_name).endswith("_arsenal_masked"):
        comparison_names.append(str(best_model_name).removesuffix("_arsenal_masked"))
    # Concatenate and filter model grids for comparison # enext
    comparison = pd.concat([state_grid, model_grid], ignore_index=True)
    comparison = comparison.loc[comparison["model"].isin(dict.fromkeys(comparison_names))]
    # Map model and dataset names to compact labels for plotting # enext
    comparison["model_label"] = comparison["model"].map(compact_model_label)
    comparison["dataset_label"] = comparison["dataset"].map({"2025_holdout": "2025 holdout", "2026_to_date": "2026 to date"})
    # Prepare data for plotting exact and top-3 accuracy # enext
    plot_df = comparison[["model_label", "dataset_label", "exact_accuracy", "top3_accuracy"]].melt(
        ["model_label", "dataset_label"], var_name="metric", value_name="score"
    )
    plot_df["metric"] = plot_df["metric"].map({"exact_accuracy": "Exact accuracy", "top3_accuracy": "Top-3 accuracy"})
    # Create subplots for exact and top-3 accuracy # enext
    fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.4), sharey=True)
    handles = labels = None
    for ax, metric in zip(axes, ["Exact accuracy", "Top-3 accuracy"]):
        sub = plot_df.loc[plot_df["metric"].eq(metric)]
        sns.barplot(
            data=sub,
            y="model_label",
            x="score",
            hue="dataset_label",
            palette=["#2f5f8f", "#d9a441"],
            errorbar=None,
            ax=ax,
        )
        ax.set_title(metric)
        ax.set_xlabel("Score")
        ax.set_ylabel("")
        ax.set_xlim(0, 1)
        if handles is None:
            handles, labels = ax.get_legend_handles_labels()
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()
    if handles and labels:
        fig.legend(handles, labels, title="", loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.03))
    fig.suptitle("Model comparison: 2025 holdout versus 2026 temporal validation", y=1.02, fontweight="bold")
    save_fig("fig2_model_comparison.png")

    # Figure 3: Regression coefficient heatmap # enext
    # Define specific regression terms to display # enext
    terms = [
        "Count: 0-2",
        "Count: 1-2",
        "Count: 2-0",
        "Count: 3-0",
        "Count: 3-2",
        "Previous pitch: FF",
        "Previous pitch: SI",
        "Previous pitch: SL",
        "Previous pitch: CH",
        "Batter side: R",
    ]
    reg = regression.copy()
    # Clean regression terms for readability # enext
    reg["term_clean"] = reg["term"].map(readable_term)
    # Filter regression results to include only specified terms # enext
    reg_sub = reg.loc[reg["term_clean"].isin(terms)].copy()
    # Pivot table to create a heatmap of beta coefficients # enext
    heat = reg_sub.pivot_table(index="term_clean", columns="target_pitch_type", values="beta", aggfunc="mean").fillna(0)
    # Reindex the heatmap to ensure specific term order # enext
    heat = heat.reindex([t for t in terms if t in heat.index])
    plt.figure(figsize=(7.2, 4.8))
    # Create a seaborn heatmap # enext
    sns.heatmap(heat, cmap="vlag", center=0, annot=True, fmt="+.2f", cbar_kws={"label": "OLS beta"})
    plt.title("Linear probability model coefficients")
    plt.xlabel("Pitch outcome")
    plt.ylabel("")
    save_fig("fig3_regression_beta_heatmap.png")

In [ ]:
def shap_interpretation(train: pd.DataFrame, test: pd.DataFrame, target_pitches: list[str]) -> pd.DataFrame:
    # Define categorical and numerical features for SHAP analysis # enext
    shap_cat = ["pitcher_name", "count", "prev_pitch_type", "prev2_pitch_type", "stand", "prev_description", "base_state"]
    shap_num = ["balls", "strikes", "pitch_number", "prev_release_speed", "score_diff_batting_team"]
    shap_cat = available_columns(train, shap_cat)
    shap_num = available_columns(train, shap_num)
    features = shap_cat + shap_num
    # Sample training and explanation data for SHAP efficiency # enext
    sample_train = train.sample(min(12000, len(train)), random_state=42)
    sample_explain = test.sample(min(800, len(test)), random_state=42)

    # Preprocessing pipeline for categorical features for SHAP # enext
    cat_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=25, sparse_output=False)),
        ]
    )
    # Preprocessing pipeline for numerical features for SHAP # enext
    num_pipe = Pipeline([("impute", SimpleImputer(strategy="median"))])
    # Combine preprocessors for SHAP features # enext
    pre = ColumnTransformer([("cat", cat_pipe, shap_cat), ("num", num_pipe, shap_num)])
    # Transform training and explanation data # enext
    X_train = pre.fit_transform(sample_train[features])
    X_explain = pre.transform(sample_explain[features])
    # Get cleaned feature names after one-hot encoding # enext
    feature_names = [clean_feature_name(name) for name in pre.get_feature_names_out()]

    all_rows = []
    # Create a figure with subplots for SHAP panels # enext
    fig, axes = plt.subplots(2, 2, figsize=(9.0, 7.0))
    axes = axes.flatten()
    # Iterate through target pitches to generate SHAP explanations # enext
    for ax, pitch in zip(axes, target_pitches[:4]):
        # Create a binary target variable for the current pitch # enext
        y = sample_train["pitch_type"].eq(pitch).astype(int)
        # Train a RandomForestClassifier for SHAP analysis # enext
        model = RandomForestClassifier(
            n_estimators=120,
            min_samples_leaf=30,
            max_features="sqrt",
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42,
        )
        model.fit(X_train, y)
        # Initialize SHAP TreeExplainer # enext
        explainer = shap.TreeExplainer(model)
        # Compute SHAP values # enext
        vals = explainer.shap_values(X_explain)
        # Extract relevant SHAP values for the positive class # enext
        if isinstance(vals, list):
            shap_vals = vals[1]
        elif getattr(vals, "ndim", 0) == 3:
            shap_vals = vals[:, :, 1]
        else:
            shap_vals = vals
        # Calculate mean absolute SHAP values for feature importance # enext
        importance = pd.DataFrame(
            {
                "target_pitch_type": pitch,
                "target_pitch_label": pitch_label(pitch),
                "feature": feature_names,
                "mean_abs_shap": np.abs(shap_vals).mean(axis=0),
            }
        ).sort_values("mean_abs_shap", ascending=False)
        all_rows.append(importance)
        # Plot top 10 features by mean absolute SHAP value # enext
        top = importance.head(10).sort_values("mean_abs_shap", ascending=True)
        ax.barh(top["feature"], top["mean_abs_shap"], color="#2f5f8f")
        ax.set_title(f"Next pitch = {pitch_label(pitch)}")
        ax.set_xlabel("Mean |SHAP|")
        ax.set_ylabel("")
    # Turn off unused subplots # enext
    for ax in axes[len(target_pitches[:4]) :]:
        ax.axis("off")
    plt.suptitle("SHAP interpretation: what shifts pitch-selection probabilities?", y=1.02, fontweight="bold")
    save_fig("fig4_shap_pitch_type_panels.png")

    # Aggregate and save SHAP results # enext
    out = pd.concat(all_rows, ignore_index=True)
    out.to_csv(TABLE_DIR / "shap_pitch_type_feature_importance.csv", index=False)
    # Calculate global feature importance # enext
    global_out = (
        out.groupby("feature", as_index=False)
        .agg(mean_abs_shap=("mean_abs_shap", "mean"), pitch_types=("target_pitch_type", "nunique"))
        .sort_values("mean_abs_shap", ascending=False)
    )
    global_out.to_csv(TABLE_DIR / "shap_global_feature_importance.csv", index=False)
    # Group features by conceptual categories # enext
    grouped_out = out.assign(feature_group=out["feature"].map(shap_feature_group))
    grouped_out = (
        grouped_out.groupby(["target_pitch_type", "target_pitch_label", "feature_group"], as_index=False)
        .agg(mean_abs_shap=("mean_abs_shap", "sum"), raw_features=("feature", "nunique"))
        .sort_values(["target_pitch_type", "mean_abs_shap"], ascending=[True, False])
    )
    grouped_out.to_csv(TABLE_DIR / "shap_grouped_feature_importance.csv", index=False)

    # Calculate grouped global feature importance # enext
    grouped_global = (
        grouped_out.groupby("feature_group", as_index=False)
        .agg(mean_abs_shap=("mean_abs_shap", "mean"), pitch_types=("target_pitch_type", "nunique"))
        .sort_values("mean_abs_shap", ascending=False)
    )
    grouped_global.to_csv(TABLE_DIR / "shap_grouped_global_feature_importance.csv", index=False)

    # Figure 5: Global SHAP feature importance # enext
    top_global = global_out.head(20).sort_values("mean_abs_shap", ascending=True)
    plt.figure(figsize=(8.2, 5.6))
    plt.barh(top_global["feature"], top_global["mean_abs_shap"], color="#2f5f8f")
    plt.title("Global SHAP feature importance across pitch outcomes")
    plt.xlabel("Mean |SHAP| across one-vs-rest pitch models")
    plt.ylabel("")
    save_fig("fig5_shap_global_feature_importance.png")

    # Figure 6: Grouped SHAP feature importance # enext
    top_grouped = grouped_global.head(14).sort_values("mean_abs_shap", ascending=True)
    plt.figure(figsize=(8.2, 4.8))
    plt.barh(top_grouped["feature_group"], top_grouped["mean_abs_shap"], color="#2f5f8f")
    plt.title("Grouped SHAP importance by baseball concept")
    plt.xlabel("Summed mean |SHAP| within feature group")
    plt.ylabel("")
    save_fig("fig6_shap_grouped_global_importance.png")
    return out

In [ ]:
def write_paper_outline(
    best_state: pd.Series,
    best_model: pd.Series,
    target_pitches: list[str],
    temporal_model: pd.Series | None = None,
) -> None:
    temporal_line = ""
    # Conditionally add temporal model results if available # enext
    if temporal_model is not None:
        temporal_line = (
            f"- On 2026-to-date temporal validation, the strongest pooled model reaches exact accuracy "
            f"{temporal_model['exact_accuracy']:.1%} and top-3 accuracy {temporal_model['top3_accuracy']:.1%}.\n"
        )
    # Construct the markdown outline for the paper # enext
    outline = f"""# Paper-Ready Results Outline\n\n## Working Title\nPitch Sequencing as Strategic Interaction: Modeling MLB Pitch Selection as Pitcher-Specific State Machines\n\n## Research Question\nHow do MLB pitchers choose their next pitch as a function of count, previous pitch, batter handedness, and pitcher-specific arsenal?\n\n## Main Model Findings\n- Best tuned state-machine specification: `{best_state['model']}`, alpha={best_state.get('alpha')}, minimum state count={best_state.get('min_count')}.\n- Best pooled predictive model: `{best_model['model']}`.\n- The best model reaches exact accuracy {best_model['exact_accuracy']:.1%}, top-2 accuracy {best_model['top2_accuracy']:.1%}, and top-3 accuracy {best_model['top3_accuracy']:.1%} on the 2025 holdout.\n{temporal_line}\n\n## Regression Strategy\nLinear probability models estimate whether the next pitch is one of: {', '.join(target_pitches)}. Predictors include count, previous pitch, batter side, and pitcher fixed effects.\n\n## Suggested Figure Captions\n**Figure 1. Pitcher-specific arsenals.** Each row shows the observed pitch menu available to a pitcher under the rule usage >= 3% or at least 30 pitches.\n\n**Figure 2. Model comparison.** State-machine and pooled tree models are evaluated against the pitch actually thrown. Top-k accuracy is emphasized because pitch calling is a mixed-strategy decision.\n\n**Figure 3. Regression coefficient heatmap.** OLS linear probability coefficients show how count, previous pitch, and batter side shift the probability of each pitch type.\n\n**Figure 4. SHAP feature importance.** One-vs-rest random forest explanations show which state/context features most influence the probability of specific pitch outcomes.\n\n**Figure 5. Global SHAP feature importance.** SHAP values are averaged across pitch-specific explanation models to identify the strongest general sequencing signals.\n\n**Figure 6. Grouped SHAP feature importance.** One-hot features are collapsed into baseball concepts so pitcher-name indicators are interpreted as pitcher identity/arsenal rather than as standalone causal explanations.\n\n## Transparency Note\nXGBoost may require OpenMP (`libomp`) on macOS. The local run uses sklearn's histogram gradient boosting when XGBoost cannot load. Colab/Linux can run XGBoost directly if desired.\n"""
    # Write the generated outline to a markdown file # enext
    (PAPER_DIR / "paper_ready_results_outline.md").write_text(outline, encoding="utf-8")

In [ ]:
def parse_args() -> argparse.Namespace:
    # Create an argument parser for command-line arguments # enext
    parser = argparse.ArgumentParser(description="Generate paper-ready state-machine, beta, model, and SHAP outputs.")
    # Add an argument for 'top-n' with a default value # enext
    parser.add_argument("--top-n", type=int, default=100)
    return parser.parse_args()

In [ ]:
def main() -> None:
    args = parse_args()
    ensure_dirs()
    set_style()

    # Load the feature data for 2025 and potentially 2026
    df_2025, df_2026 = load_features(args.top_n)

    # Prepare the 2025 data for modeling
    train_2025 = model_frame(df_2025)

    # Determine the unique pitch types to be used as classes
    keep_pitches = pd.Index(sorted(train_2025["pitch_type"].unique()))

    # Prepare the 2026 evaluation data, filtering by `keep_pitches`
    eval_2026 = model_frame(df_2026, keep_pitches) if not df_2026.empty else pd.DataFrame()

    # Split the 2025 data into training and testing sets, ensuring stratification if possible
    stratify = train_2025["pitch_type"] if train_2025["pitch_type"].value_counts().min() >= 2 else None
    train_rows, test_rows = train_test_split(
        train_2025,
        test_size=0.20,
        random_state=42,
        stratify=stratify,
    )

    # Get the unique classes (pitch types) for classification models
    classes = np.asarray(sorted(train_rows["pitch_type"].unique()))

    # Identify the top 5 most frequent pitch types for regression and SHAP analysis
    top_targets = train_rows["pitch_type"].value_counts().head(5).index.tolist()

    log("Building arsenals...")
    # Build pitcher arsenals based on training data
    arsenal = build_arsenal(train_rows)
    # Create a dictionary mapping pitchers to their allowed pitches
    allowed = arsenal_map(arsenal)

    log("Running paper-style OLS linear probability models before boosted prediction models...")
    # Fit ordinary least squares (OLS) regression models
    regression = fit_regression_tables(train_rows, top_targets)

    log("Tuning state-machine parameters...")
    # Tune and evaluate state-machine models
    state_grid = tune_state_machine(train_rows, test_rows, classes, allowed, eval_2026)

    log("Tuning pooled prediction models as a benchmark to the per-pitcher loop.")
    # Tune and evaluate pooled prediction models (e.g., HGB)
    model_grid = tune_models(train_rows, test_rows, classes, allowed, eval_2026)

    log("Generating polished figures...")
    # Generate paper-ready figures using the results
    make_paper_figures(arsenal, state_grid, model_grid, regression)

    log("Computing intuitive SHAP panels...")
    # Perform SHAP interpretation to explain model predictions
    shap_df = shap_interpretation(train_rows, test_rows, top_targets[:4])

    # Extract the best performing state-machine and pooled models from the tuning grids
    best_state = state_grid.loc[state_grid["dataset"].eq("2025_holdout")].iloc[0]
    best_model = model_grid.loc[model_grid["dataset"].eq("2025_holdout")].iloc[0]
    best_temporal = None
    if not model_grid.loc[model_grid["dataset"].eq("2026_to_date")].empty:
        best_temporal = model_grid.loc[model_grid["dataset"].eq("2026_to_date")].iloc[0]

    # Combine and save the top models for comparison
    combined = pd.concat(
        [
            state_grid.groupby("dataset", group_keys=False).head(12),
            model_grid.groupby("dataset", group_keys=False).head(12),
        ],
        ignore_index=True,
    )
    combined.to_csv(TABLE_DIR / "paper_model_comparison_top_models.csv", index=False)

    # Write out a markdown outline of the paper results
    write_paper_outline(best_state, best_model, top_targets, best_temporal)

    # Create a summary dictionary of key results
    summary = {
        "n_2025_rows": int(len(train_2025)),
        "n_2026_rows": int(len(eval_2026)) if not eval_2026.empty else 0,
        "target_pitches": top_targets,
        "best_state_machine": best_state.to_dict(),
        "best_pooled_model": best_model.to_dict(),
        "best_2026_temporal_model": best_temporal.to_dict() if best_temporal is not None else None,
    }
    # Save the summary as a JSON file
    (PAPER_DIR / "paper_ready_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

    # Log and print the best state-machine model results
    log("\n=== Best state-machine model ===")
    print(state_grid.loc[state_grid["dataset"].eq("2025_holdout")].head(5).to_string(index=False), flush=True)

    # Log and print the best pooled models results
    log("\n=== Best pooled models ===")
    print(model_grid.loc[model_grid["dataset"].eq("2025_holdout")].head(8).to_string(index=False), flush=True)

    # Log and print 2026 temporal validation results if available
    if not eval_2026.empty:
        log("\n=== 2026 temporal validation ===")
        temporal = pd.concat(
            [
                state_grid.loc[state_grid["dataset"].eq("2026_to_date")].head(4),
                model_grid.loc[model_grid["dataset"].eq("2026_to_date")].head(8),
            ],
            ignore_index=True,
        )
        print(
            temporal[
                ["model", "dataset", "rows", "exact_accuracy", "top2_accuracy", "top3_accuracy", "macro_f1", "log_loss"]
            ].to_string(index=False),
            flush=True,
        )

    # Log and print the top SHAP features
    log("\n=== Top SHAP features ===")
    print(shap_df.groupby("target_pitch_type").head(5).to_string(index=False), flush=True)

    # Log the path to the paper-ready output folder
    log("\n=== Paper-ready folder ===")
    log(str(PAPER_DIR.relative_to(ROOT)))

In [ ]:
if __name__ == "__main__": # enext
    # Call the main function when the script is executed # enext
    main()